# Phase 3 — Join Pollution + Disease Data
### Notebook: `phase_3_join.ipynb`

**What this notebook does:**
1. Loads `outputs/air_pollution_master.csv` (city-level, monthly)
2. Loads `outputs/disease_master.csv` (state-level, annual)
3. Aggregates pollution from city × month → state × year
4. Joins the two datasets on state name + year
5. Saves `outputs/combined_master.csv` — the main analysis table for the project

---
**The fundamental mismatch being solved here:**  
Pollution data = city level, monthly.  
Disease data = state level, annual.  
We cannot join them directly. We must first collapse pollution to state × year level.

---
**Input files:**
- `outputs/air_pollution_master.csv`  (produced by Phase 1)
- `outputs/disease_master.csv`        (produced by Phase 2)

**Output file:**
- `outputs/combined_master.csv`

---
**Confirmed structure of air_pollution_master.csv (from Phase 1 code):**
- Identity cols: `state`, `state_original`, `city`, `year`, `month`
- Quality cols: `data_quality_tier`, `actual_data_years`, `data_first_year`, `data_last_year`, `n_stations`
- For each of 29 measurement columns: `{col}_mean`, `{col}_flag`, `{col}_coverage_pct`
- Pollutant naming example: `PM2.5 (ug/m3)_mean`, `CO (mg/m3)_mean`, etc.

**Confirmed structure of disease_master.csv (from Phase 2):**
- `location_name`, `year`, `cause_name`, `metric_id`, `metric_name`, `val`, `upper`, `lower`, `evidence_strength`, `source_file`

---
## Cell 1 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported")

Libraries imported


---
## Cell 2 — Set Paths and Load Both Files

Both input files were produced by earlier phases and live in `outputs/`.  
We load them and immediately verify their shapes match what we expect.

In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path('..')
OUTPUT_DIR = BASE_DIR / 'outputs'

POLLUTION_MASTER_PATH = OUTPUT_DIR / 'air_pollution_master.csv'
DISEASE_MASTER_PATH   = OUTPUT_DIR / 'disease_master.csv'

# ── Verify both files exist before loading ────────────────────────────────────
for fpath in [POLLUTION_MASTER_PATH, DISEASE_MASTER_PATH]:
    if not fpath.exists():
        raise FileNotFoundError(
            f"File not found: {fpath}\n"
            "Run Phase 1 and Phase 2 first to produce these files."
        )

# ── Load pollution master (city x month) ─────────────────────────────────────
# low_memory=False: prevents dtype guessing warnings on the many _mean columns
df_pollution = pd.read_csv(POLLUTION_MASTER_PATH, low_memory=False)
print(f"air_pollution_master.csv loaded")
print(f"   Shape : {df_pollution.shape[0]:,} rows x {df_pollution.shape[1]} columns")
print(f"   Cities: {df_pollution['city'].nunique()}")
print(f"   States: {df_pollution['state'].nunique()}")
print(f"   Years : {df_pollution['year'].min()} to {df_pollution['year'].max()}")
print()

# ── Load disease master (state x year) ───────────────────────────────────────
df_disease = pd.read_csv(DISEASE_MASTER_PATH)
print(f"disease_master.csv loaded")
print(f"   Shape : {df_disease.shape[0]:,} rows x {df_disease.shape[1]} columns")
print(f"   States: {df_disease['location_name'].nunique()}")
print(f"   Years : {df_disease['year'].min()} to {df_disease['year'].max()}")
print(f"   Causes: {df_disease['cause_name'].nunique()}")

air_pollution_master.csv loaded
   Shape : 10,295 rows x 97 columns
   Cities: 241
   States: 30
   Years : 2010 to 2023

disease_master.csv loaded
   Shape : 18,228 rows x 10 columns
   States: 31
   Years : 2010 to 2023
   Causes: 21


---
## Cell 3 — Inspect and Confirm Join Keys

The join in Phase 3 connects rows using two keys: **state name + year**.  

In `air_pollution_master.csv` the state column is called **`state`**.  
In `disease_master.csv` the state column is called **`location_name`**.  
Both were standardised to GBD names in their respective phases.  

This cell confirms the names actually match before we attempt the join.  
If they don't match here, the join will silently produce NaN rows — not a crash.

In [4]:
# ── Get the unique state names from each dataset ──────────────────────────────
pollution_states = set(df_pollution['state'].unique())
disease_states   = set(df_disease['location_name'].unique())

print(f"States in pollution master : {len(pollution_states)}")
print(f"States in disease master   : {len(disease_states)}")
print()

# ── Find any mismatches ───────────────────────────────────────────────────────
# WHY: A mismatch means that state's data will be lost in the join — not crashed,
# just silently dropped. We catch it here before that happens.
only_in_pollution = pollution_states - disease_states
only_in_disease   = disease_states - pollution_states
matched           = pollution_states & disease_states

print(f"States that will join correctly : {len(matched)}")

if only_in_pollution:
    print(f"\nIn pollution but NOT in disease ({len(only_in_pollution)}) — these states will get NaN disease values:")
    for s in sorted(only_in_pollution):
        print(f"   '{s}'")
else:
    print("All pollution states found in disease data")

if only_in_disease:
    print(f"\nIn disease but NOT in pollution ({len(only_in_disease)}) — these states will get NaN pollution values:")
    for s in sorted(only_in_disease):
        print(f"   '{s}'  (known: Goa has no monitoring stations)")
else:
    print("All disease states found in pollution data")

print()

# ── Check year overlap ────────────────────────────────────────────────────────
# WHY: We can only have a complete join for years that exist in BOTH datasets.
# Disease data: 2010-2023. Pollution data: varies by station.
pollution_years = set(df_pollution['year'].unique())
disease_years   = set(df_disease['year'].unique())
shared_years    = sorted(pollution_years & disease_years)

print(f"Years in pollution : {sorted(pollution_years)[0]} to {sorted(pollution_years)[-1]}")
print(f"Years in disease   : {sorted(disease_years)[0]} to {sorted(disease_years)[-1]}")
print(f"Shared years       : {shared_years[0]} to {shared_years[-1]} ({len(shared_years)} years)")

States in pollution master : 30
States in disease master   : 31

States that will join correctly : 30
All pollution states found in disease data

In disease but NOT in pollution (1) — these states will get NaN pollution values:
   'Goa'  (known: Goa has no monitoring stations)

Years in pollution : 2010 to 2023
Years in disease   : 2010 to 2023
Shared years       : 2010 to 2023 (14 years)


---
## Cell 4 — Identify Pollutant Mean Columns

Phase 1 created columns named `{pollutant}_mean` for every measurement column.  
We discover them dynamically (not hardcoded) so this cell works correctly  
even if Phase 1 added or removed columns in a future run.

We split them into:
- **Pollutant means** — the core air quality readings (PM2.5, NO2, SO2 etc.)
- **Weather means** — meteorological readings (wind speed, temperature etc.)

Both groups will be aggregated to state-year level. Weather is important for  
Phase 5 Task A (predicting pollution from weather patterns).

In [5]:
# ── Find all _mean columns dynamically ───────────────────────────────────────
# WHY: This is safer than a hardcoded list. If Phase 1 changes what columns it
# saves, this cell automatically adapts — no manual update needed here.
ALL_MEAN_COLS = [c for c in df_pollution.columns if c.endswith('_mean')]

# ── Separate pollutant from weather mean columns ──────────────────────────────
# Weather column base names — confirmed from Phase 1 WEATHER_COLS list
WEATHER_BASE = ['WS (m/s)', 'WD (degree)', 'RH (%)', 'AT (degree C)',
                'SR (W/mt2)', 'RF (mm)', 'BP (mmHg)']
WEATHER_MEAN_COLS    = [f'{c}_mean' for c in WEATHER_BASE if f'{c}_mean' in df_pollution.columns]
POLLUTANT_MEAN_COLS  = [c for c in ALL_MEAN_COLS if c not in WEATHER_MEAN_COLS]

print(f"Total _mean columns found in pollution master: {len(ALL_MEAN_COLS)}")
print()
print(f"Pollutant mean columns ({len(POLLUTANT_MEAN_COLS)}):")
for c in POLLUTANT_MEAN_COLS:
    print(f"   {c}")
print()
print(f"Weather mean columns ({len(WEATHER_MEAN_COLS)}):")
for c in WEATHER_MEAN_COLS:
    print(f"   {c}")

Total _mean columns found in pollution master: 29

Pollutant mean columns (22):
   PM2.5 (ug/m3)_mean
   PM10 (ug/m3)_mean
   NO (ug/m3)_mean
   NO2 (ug/m3)_mean
   NOx (ppb)_mean
   SO2 (ug/m3)_mean
   CO (mg/m3)_mean
   NH3 (ug/m3)_mean
   Ozone (ug/m3)_mean
   Benzene (ug/m3)_mean
   Toluene (ug/m3)_mean
   Eth-Benzene (ug/m3)_mean
   MP-Xylene (ug/m3)_mean
   O Xylene (ug/m3)_mean
   Xylene (ug/m3)_mean
   CH4 (ug/m3)_mean
   NMHC (ug/m3)_mean
   THC (ug/m3)_mean
   CO2 (mg/m3)_mean
   HCHO (ug/m3)_mean
   Hg (ug/m3)_mean
   SPM (ug/m3)_mean

Weather mean columns (7):
   WS (m/s)_mean
   WD (degree)_mean
   RH (%)_mean
   AT (degree C)_mean
   SR (W/mt2)_mean
   RF (mm)_mean
   BP (mmHg)_mean


---
## Cell 5 — Aggregate Pollution: City × Month → State × Year

This is the core step that makes the join possible.

**What this aggregation does:**  
For each (state, year), take ALL city-month rows and compute the mean of each `_mean` column.

**Why NaN handling is automatic here:**  
In Phase 1, any city-month with less than 25% hourly coverage was assigned `NaN` as its `_mean` value and flagged `insufficient`. When we compute `mean()` across city-months, pandas skips NaN values by default (`skipna=True`). So months with insufficient data are automatically excluded from the state average — we never average fake values.

**What `n_city_months` tells us:**  
We also count how many city-month rows actually contributed to each state-year. A state with 10 cities each having 12 months = 120 city-months. A state with 2 cities each having 3 months = 6. Higher is more reliable.

In [6]:
print("Aggregating city-monthly pollution → state-annual averages...")
print("(This may take a moment — computing mean across all city-months per state-year)")
print()

# ── Step 1: Compute annual mean for every pollutant and weather column ─────────
# groupby(['state', 'year']): groups all rows that share the same state AND year
# [ALL_MEAN_COLS].mean(): for each group, compute the mean of every _mean column
# skipna=True is the pandas default — NaN city-months are automatically excluded
pollution_agg = (
    df_pollution
    .groupby(['state', 'year'], sort=True)[ALL_MEAN_COLS]
    .mean()
    .reset_index()
)

# ── Step 2: Count how many city-months contributed to each state-year ──────────
# WHY: This is quality metadata. A state with 20 contributing city-months is
# more reliable than one with only 2. We attach this to the output.
city_month_counts = (
    df_pollution
    .groupby(['state', 'year'], sort=True)
    .agg(
        n_city_months    = ('city',  'count'),    # total city-month rows
        n_cities         = ('city',  'nunique'),  # distinct cities that had data
    )
    .reset_index()
)

# ── Step 3: Also track dominant tier per state-year ───────────────────────────
# WHY: In Phase 5 we need to know which state-years have high-quality data.
# We take the BEST (lowest number) tier present among cities for that state-year.
# E.g. if Delhi has tier_1 and tier_2 cities, the state-year is tagged tier_1.

# Map tier string to number so we can find the minimum (best) tier
TIER_TO_NUM = {'tier_1': 1, 'tier_2': 2, 'tier_3': 3, 'tier_4': 4}
NUM_TO_TIER = {v: k for k, v in TIER_TO_NUM.items()}

df_pollution['tier_num'] = df_pollution['data_quality_tier'].map(TIER_TO_NUM)

tier_per_state_year = (
    df_pollution
    .groupby(['state', 'year'], sort=True)['tier_num']
    .min()    # best (lowest number) tier among all cities in this state-year
    .reset_index()
)
tier_per_state_year['best_city_tier'] = (
    tier_per_state_year['tier_num'].map(NUM_TO_TIER)
)
tier_per_state_year.drop(columns=['tier_num'], inplace=True)

# Clean up the helper column from df_pollution
df_pollution.drop(columns=['tier_num'], inplace=True)

# ── Step 4: Merge the three aggregated pieces together ────────────────────────
pollution_agg = pollution_agg.merge(city_month_counts,    on=['state', 'year'], how='left')
pollution_agg = pollution_agg.merge(tier_per_state_year,  on=['state', 'year'], how='left')

print(f"State-annual pollution aggregation complete")
print(f"   Shape: {pollution_agg.shape[0]:,} rows x {pollution_agg.shape[1]} columns")
print(f"   States: {pollution_agg['state'].nunique()}")
print(f"   Years : {pollution_agg['year'].min()} to {pollution_agg['year'].max()}")
print()
print("Sample — first 3 rows (showing identity + first 3 pollutant columns):")
sample_cols = ['state', 'year', 'n_cities', 'n_city_months', 'best_city_tier'] + ALL_MEAN_COLS[:3]
print(pollution_agg[sample_cols].head(3).to_string())

Aggregating city-monthly pollution → state-annual averages...
(This may take a moment — computing mean across all city-months per state-year)

State-annual pollution aggregation complete
   Shape: 229 rows x 34 columns
   States: 30
   Years : 2010 to 2023

Sample — first 3 rows (showing identity + first 3 pollutant columns):
            state  year  n_cities  n_city_months best_city_tier  PM2.5 (ug/m3)_mean  PM10 (ug/m3)_mean  NO (ug/m3)_mean
0  Andhra Pradesh  2016         1              6         tier_1           31.736268          62.194140        21.775029
1  Andhra Pradesh  2017         5             32         tier_1           43.869432          82.603280        14.192763
2  Andhra Pradesh  2018         5             60         tier_1           38.462310          83.100718         9.753432


---
## Cell 6 — Verify State-Year Coverage Before Joining

Before the join, we check how complete the pollution aggregation is.  
We want to know: for how many (state, year) combinations do we have PM2.5 data?  

PM2.5 is used as the proxy for overall data quality because it's the most  
consistently measured pollutant across all stations (99.1% station coverage).

In [7]:
PM25_MEAN_COL = 'PM2.5 (ug/m3)_mean'

print("=" * 55)
print("POLLUTION AGGREGATION COVERAGE CHECK")
print("=" * 55)

total_state_years   = len(pollution_agg)
has_pm25            = pollution_agg[PM25_MEAN_COL].notna().sum()
missing_pm25        = total_state_years - has_pm25

print(f"Total state-year combinations : {total_state_years:,}")
print(f"Have PM2.5 data               : {has_pm25:,}  ({has_pm25/total_state_years*100:.1f}%)")
print(f"Missing PM2.5 data            : {missing_pm25:,}  ({missing_pm25/total_state_years*100:.1f}%)")
print()

# ── Show PM2.5 coverage per state (how many years of data each state has) ─────
print("PM2.5 data availability per state (years with data / years possible):")
total_years_possible = pollution_agg['year'].nunique()

state_coverage = (
    pollution_agg
    .groupby('state')[PM25_MEAN_COL]
    .apply(lambda x: x.notna().sum())
    .reset_index()
    .rename(columns={PM25_MEAN_COL: 'years_with_pm25'})
    .sort_values('years_with_pm25', ascending=False)
)

for _, row in state_coverage.iterrows():
    bar_len = int(row['years_with_pm25'] / total_years_possible * 20)
    bar     = '█' * bar_len + '░' * (20 - bar_len)
    print(f"   {row['state']:35s} {bar}  {int(row['years_with_pm25'])}/{total_years_possible} yrs")

print()

# ── Best tier distribution across state-years ─────────────────────────────────
print("Best city tier distribution across all state-years:")
tier_dist = pollution_agg['best_city_tier'].value_counts().sort_index()
for tier, count in tier_dist.items():
    print(f"   {tier}: {count:,} state-years")

POLLUTION AGGREGATION COVERAGE CHECK
Total state-year combinations : 229
Have PM2.5 data               : 168  (73.4%)
Missing PM2.5 data            : 61  (26.6%)

PM2.5 data availability per state (years with data / years possible):
   Uttar Pradesh                       ██████████████░░░░░░  10/14 yrs
   Bihar                               ████████████░░░░░░░░  9/14 yrs
   Tamil Nadu                          ████████████░░░░░░░░  9/14 yrs
   Rajasthan                           ████████████░░░░░░░░  9/14 yrs
   Maharashtra                         ████████████░░░░░░░░  9/14 yrs
   Haryana                             ████████████░░░░░░░░  9/14 yrs
   Andhra Pradesh                      ███████████░░░░░░░░░  8/14 yrs
   Punjab                              ██████████░░░░░░░░░░  7/14 yrs
   Odisha                              ██████████░░░░░░░░░░  7/14 yrs
   Telangana                           ██████████░░░░░░░░░░  7/14 yrs
   Madhya Pradesh                      ██████████░░░░░░░░░░  7/14 

---
## Cell 7 — Perform the Join

We join the state-annual pollution data with the state-annual disease data.

**Join type: left join from disease to pollution**  
- Disease is the left table. Every disease row is kept.
- Pollution is the right table. If a state-year has no pollution data, those columns get NaN.
- Goa (which has no pollution stations) will appear with NaN pollution columns — correct.

**Shape after join:**  
The disease table has 18,228 rows (31 states × 14 years × 21 causes × 2 metrics, minus gaps).  
Each of those rows gets pollution columns added. The row count stays ~18,228.  
This is the correct long format — one row per (state, year, disease cause, metric type).

In [8]:
# ── Rename pollution 'state' column to 'location_name' for the join ───────────
# WHY: The join key must have the same column name in both DataFrames.
# We rename the pollution side to match the disease side's 'location_name'.
# This keeps the final output consistent — one column, one name, no ambiguity.
pollution_agg_for_join = pollution_agg.rename(columns={'state': 'location_name'})

# ── Perform the join ──────────────────────────────────────────────────────────
# how='left': keep ALL disease rows. Add pollution data where available.
# on=['location_name', 'year']: match rows where BOTH state name AND year match.
# WHY left join and not inner: we want to preserve Goa's disease data (even
# though it has no pollution stations), and we want to see which state-years
# have disease data but no pollution data — inner join would hide those silently.
df_combined = df_disease.merge(
    pollution_agg_for_join,
    on=['location_name', 'year'],
    how='left'
)

print(f"Join complete")
print(f"   Disease rows (left table)  : {len(df_disease):,}")
print(f"   Pollution state-years (right): {len(pollution_agg_for_join):,}")
print(f"   Combined rows after join   : {len(df_combined):,}")
print()

# ── Check how many rows got pollution data vs stayed NaN ─────────────────────
# We check PM2.5 as the proxy for 'did this row get pollution data'
has_pollution = df_combined[PM25_MEAN_COL].notna().sum()
no_pollution  = df_combined[PM25_MEAN_COL].isna().sum()

print(f"Rows with pollution data    : {has_pollution:,}  ({has_pollution/len(df_combined)*100:.1f}%)")
print(f"Rows without pollution data : {no_pollution:,}  ({no_pollution/len(df_combined)*100:.1f}%)")
print()

# ── Which states/years had no matching pollution data? ────────────────────────
# WHY: We want to know exactly what was lost, not just a number.
unmatched_states = (
    df_combined[df_combined[PM25_MEAN_COL].isna()]
    ['location_name']
    .unique()
)
if len(unmatched_states) > 0:
    print(f"States with NO pollution data in combined output:")
    for s in sorted(unmatched_states):
        print(f"   '{s}'")
    print("   (These states have disease data but no monitoring stations)")
print()

# ── Show sample of combined output ────────────────────────────────────────────
sample_show = ['location_name', 'year', 'cause_name', 'metric_name',
               'val', 'evidence_strength', 'n_cities', PM25_MEAN_COL]
sample_show = [c for c in sample_show if c in df_combined.columns]
print("Sample rows from combined_master (first 4):")
print(df_combined[sample_show].head(4).to_string())

Join complete
   Disease rows (left table)  : 18,228
   Pollution state-years (right): 229
   Combined rows after join   : 18,228

Rows with pollution data    : 7,056  (38.7%)
Rows without pollution data : 11,172  (61.3%)

States with NO pollution data in combined output:
   'Andhra Pradesh'
   'Arunachal Pradesh'
   'Assam'
   'Bihar'
   'Chhattisgarh'
   'Delhi'
   'Goa'
   'Gujarat'
   'Haryana'
   'Himachal Pradesh'
   'Jammu & Kashmir and Ladakh'
   'Jharkhand'
   'Karnataka'
   'Kerala'
   'Madhya Pradesh'
   'Maharashtra'
   'Manipur'
   'Meghalaya'
   'Mizoram'
   'Nagaland'
   'Odisha'
   'Other Union Territories'
   'Punjab'
   'Rajasthan'
   'Sikkim'
   'Tamil Nadu'
   'Telangana'
   'Tripura'
   'Uttar Pradesh'
   'Uttarakhand'
   'West Bengal'
   (These states have disease data but no monitoring stations)

Sample rows from combined_master (first 4):
    location_name  year                       cause_name metric_name          val evidence_strength  n_cities  PM2.5 (ug/m3)_

---
## Cell 8 — Add a Join Quality Flag Column

We add a `join_quality` column to every row so analysts know at a glance  
how reliable each row's pollution-disease link is.

| join_quality | Meaning |
|---|---|
| `full` | Both pollution and disease data present — safe for correlation analysis |
| `disease_only` | Disease data present but no pollution data (e.g. Goa) — disease stats valid, pollution columns NaN |
| `sparse_pollution` | Pollution data exists but fewer than 3 cities contributed — treat with caution |

In [14]:
# ── Assign join_quality to every row ─────────────────────────────────────────
# WHY: When doing correlation analysis in later phases, analysts need a fast
# way to filter to reliable rows. This column makes that one-line filter easy.

def assign_join_quality(row):
    """
    WHY THIS FUNCTION EXISTS:
    Assigns a quality label to each row based on:
    1. Whether PM2.5 data is present (proxy for 'did pollution data join')
    2. Whether n_cities is large enough to trust the state average
    """
    pm25_val  = row.get(PM25_MEAN_COL)
    n_cities  = row.get('n_cities', 0)

    if pd.isna(pm25_val):
        # No pollution data at all for this state-year
        return 'disease_only'
    elif n_cities < 3:
        # Pollution data exists, but only 1-2 cities — state average is shaky
        return 'sparse_pollution'
    else:
        # Both present and reasonable number of cities
        return 'full'

# Apply the function to every row
# axis=1 means: apply the function to each ROW (not each column)
df_combined['join_quality'] = df_combined.apply(assign_join_quality, axis=1)

# ── Report distribution ───────────────────────────────────────────────────────
print("join_quality distribution:")
for label, count in df_combined['join_quality'].value_counts().items():
    pct = count / len(df_combined) * 100
    print(f"   {label:25s}: {count:,} rows ({pct:.1f}%)")
print()
print("For Phase 5 ML and correlation analysis:")
print("   Use only rows where join_quality == 'full'")
print("   Filter: df_combined[df_combined['join_quality'] == 'full']")

join_quality distribution:
   disease_only             : 11,172 rows (61.3%)
   full                     : 4,032 rows (22.1%)
   sparse_pollution         : 3,024 rows (16.6%)

For Phase 5 ML and correlation analysis:
   Use only rows where join_quality == 'full'
   Filter: df_combined[df_combined['join_quality'] == 'full']


---
## Cell 9 — Reorder Columns for Readability

After the join, columns are in the order pandas happened to place them.  
We reorder so the output CSV reads logically:  
**identity → disease → join quality → pollutants → weather → minor**

In [10]:
# ── Define desired column order ───────────────────────────────────────────────

# Disease identity columns (always first — these are the join keys)
DISEASE_COLS = [
    'location_name',     # State name (the join key)
    'year',              # Year (the join key)
    'cause_name',        # Disease name (e.g. COPD)
    'metric_id',         # 1=Number, 3=Rate
    'metric_name',       # 'Number' or 'Rate'
    'val',               # Death count or rate per 100k
    'upper',             # Uncertainty bound
    'lower',             # Uncertainty bound
    'evidence_strength', # direct / strong / indirect
    'source_file',       # Which disease CSV this came from
]

# Pollution quality / metadata columns (tell us how reliable the pollution avg is)
POLLUTION_META_COLS = [
    'join_quality',      # full / disease_only / sparse_pollution
    'n_cities',          # How many cities contributed to the state average
    'n_city_months',     # Total city-month rows used in the average
    'best_city_tier',    # Best data quality tier among contributing cities
]

# Pollution mean columns — ALL_MEAN_COLS in their natural order
# (already split into POLLUTANT_MEAN_COLS and WEATHER_MEAN_COLS in Cell 4)
POLLUTANT_ORDER = POLLUTANT_MEAN_COLS + WEATHER_MEAN_COLS

# ── Build final column list from what actually exists ─────────────────────────
# WHY: We use [c for c if c in df_combined] instead of just reindex, to avoid
# crashing if Phase 1 didn't produce all expected columns
all_desired = DISEASE_COLS + POLLUTION_META_COLS + POLLUTANT_ORDER
final_cols  = [c for c in all_desired if c in df_combined.columns]

# ── Check if any expected columns are missing ────────────────────────────────
missing = [c for c in all_desired if c not in df_combined.columns]
if missing:
    print(f"Note: these expected columns were not found and will be skipped: {missing}")

# ── Apply ordering ────────────────────────────────────────────────────────────
df_combined = df_combined[final_cols].copy()

# Sort: state → year → cause → metric type (same as disease_master.csv sort)
df_combined.sort_values(
    ['location_name', 'year', 'cause_name', 'metric_id'],
    inplace=True
)
df_combined.reset_index(drop=True, inplace=True)

print(f"Final combined DataFrame: {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")
print()
print("Column order in combined_master.csv:")
for i, col in enumerate(df_combined.columns):
    print(f"   [{i:2d}] {col}")

Final combined DataFrame: 18,228 rows x 43 columns

Column order in combined_master.csv:
   [ 0] location_name
   [ 1] year
   [ 2] cause_name
   [ 3] metric_id
   [ 4] metric_name
   [ 5] val
   [ 6] upper
   [ 7] lower
   [ 8] evidence_strength
   [ 9] source_file
   [10] join_quality
   [11] n_cities
   [12] n_city_months
   [13] best_city_tier
   [14] PM2.5 (ug/m3)_mean
   [15] PM10 (ug/m3)_mean
   [16] NO (ug/m3)_mean
   [17] NO2 (ug/m3)_mean
   [18] NOx (ppb)_mean
   [19] SO2 (ug/m3)_mean
   [20] CO (mg/m3)_mean
   [21] NH3 (ug/m3)_mean
   [22] Ozone (ug/m3)_mean
   [23] Benzene (ug/m3)_mean
   [24] Toluene (ug/m3)_mean
   [25] Eth-Benzene (ug/m3)_mean
   [26] MP-Xylene (ug/m3)_mean
   [27] O Xylene (ug/m3)_mean
   [28] Xylene (ug/m3)_mean
   [29] CH4 (ug/m3)_mean
   [30] NMHC (ug/m3)_mean
   [31] THC (ug/m3)_mean
   [32] CO2 (mg/m3)_mean
   [33] HCHO (ug/m3)_mean
   [34] Hg (ug/m3)_mean
   [35] SPM (ug/m3)_mean
   [36] WS (m/s)_mean
   [37] WD (degree)_mean
   [38] RH (%)_mean
 

---
## Cell 10 — Quality Checks

Five checks before saving. These confirm the join worked correctly  
and the output is structurally sound for Phase 5 analysis.

In [11]:
print("=" * 60)
print("PHASE 3 QUALITY CHECKS")
print("=" * 60)

# ── Check 1: Row count ────────────────────────────────────────────────────────
print("\n[CHECK 1] Row count")
print(f"   Combined rows : {len(df_combined):,}")
print(f"   Disease rows  : {len(df_disease):,}")
if len(df_combined) == len(df_disease):
    print("   OK: Row count matches disease master — no row duplication in join")
elif len(df_combined) > len(df_disease):
    print(f"   WARNING: {len(df_combined) - len(df_disease):,} extra rows — possible duplicate state-years in pollution aggregation")
    print("   Check Cell 5 — pollution_agg should have only one row per (state, year)")
else:
    print(f"   WARNING: {len(df_disease) - len(df_combined):,} fewer rows than disease master")

# ── Check 2: NaN in disease key columns ──────────────────────────────────────
print("\n[CHECK 2] NaN in disease key columns (these should all be 0)")
for col in ['location_name', 'year', 'cause_name', 'val']:
    n_nan = df_combined[col].isna().sum()
    status = "OK" if n_nan == 0 else "WARNING"
    print(f"   [{status}]  {col:25s}: {n_nan:,} NaN values")

# ── Check 3: Duplicate state-year-cause-metric combinations ──────────────────
# WHY: If the join accidentally created duplicates, analysis will double-count deaths
print("\n[CHECK 3] Duplicate rows check")
dupe_key = ['location_name', 'year', 'cause_name', 'metric_id']
n_dupes = df_combined.duplicated(subset=dupe_key).sum()
if n_dupes == 0:
    print(f"   OK: No duplicate (state, year, cause, metric) combinations")
else:
    print(f"   WARNING: {n_dupes:,} duplicate combinations found")
    print(f"   This means the pollution aggregation had duplicate state-year rows")
    print(f"   Fix: go back to Cell 5 and investigate pollution_agg")

# ── Check 4: Join quality distribution ───────────────────────────────────────
print("\n[CHECK 4] Join quality distribution")
for label, count in df_combined['join_quality'].value_counts().items():
    pct = count / len(df_combined) * 100
    print(f"   {label:25s}: {count:,} rows ({pct:.1f}%)")

# ── Check 5: PM2.5 value sanity check ────────────────────────────────────────
# Typical urban Indian PM2.5 annual means: 30-150 ug/m3
# Values < 5 or > 500 would be suspicious
print("\n[CHECK 5] PM2.5 value range sanity check")
pm25_vals = df_combined[PM25_MEAN_COL].dropna()
if len(pm25_vals) > 0:
    print(f"   PM2.5 min  : {pm25_vals.min():.2f} ug/m3")
    print(f"   PM2.5 mean : {pm25_vals.mean():.2f} ug/m3")
    print(f"   PM2.5 max  : {pm25_vals.max():.2f} ug/m3")
    if pm25_vals.max() > 500:
        print(f"   WARNING: Max PM2.5 > 500 ug/m3 — check for unit conversion errors in Phase 1")
    elif pm25_vals.min() < 0:
        print(f"   WARNING: Negative PM2.5 values found — check raw data")
    else:
        print(f"   OK: Values in plausible range for Indian urban air quality")
else:
    print("   No PM2.5 values found")

print()
print("=" * 60)

PHASE 3 QUALITY CHECKS

[CHECK 1] Row count
   Combined rows : 18,228
   Disease rows  : 18,228
   OK: Row count matches disease master — no row duplication in join

[CHECK 2] NaN in disease key columns (these should all be 0)
   [OK]  location_name            : 0 NaN values
   [OK]  year                     : 0 NaN values
   [OK]  cause_name               : 0 NaN values
   [OK]  val                      : 0 NaN values

[CHECK 3] Duplicate rows check
   OK: No duplicate (state, year, cause, metric) combinations

[CHECK 4] Join quality distribution
   disease_only             : 11,172 rows (61.3%)
   full                     : 4,032 rows (22.1%)
   sparse_pollution         : 3,024 rows (16.6%)

[CHECK 5] PM2.5 value range sanity check
   PM2.5 min  : 6.92 ug/m3
   PM2.5 mean : 59.81 ug/m3
   PM2.5 max  : 242.63 ug/m3
   OK: Values in plausible range for Indian urban air quality



---
## Cell 11 — Save combined_master.csv

In [12]:
combined_path = OUTPUT_DIR / 'combined_master.csv'
df_combined.to_csv(combined_path, index=False)

print(f"combined_master.csv saved")
print(f"   Path  : {combined_path}")
print(f"   Shape : {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")

# ── Estimate file size ────────────────────────────────────────────────────────
import os
size_mb = os.path.getsize(combined_path) / 1e6
print(f"   Size  : {size_mb:.2f} MB")

combined_master.csv saved
   Path  : ..\outputs\combined_master.csv
   Shape : 18,228 rows x 43 columns
   Size  : 6.10 MB


---
## Cell 12 — Phase 3 Summary and Next Steps

In [13]:
print("=" * 60)
print("PHASE 3 COMPLETE — SUMMARY")
print("=" * 60)
print()

print("File saved:")
print(f"  outputs/combined_master.csv")
print(f"      {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")
print()

print("What each row in combined_master.csv represents:")
print("  One (state, year, disease cause, metric type) combination")
print("  with pollution annual averages joined in from air_pollution_master.csv")
print()

print("Key columns for analysis:")
print("  location_name   — state name (join key)")
print("  year            — year (join key)")
print("  cause_name      — disease (e.g. COPD, Ischemic heart disease)")
print("  metric_name     — 'Number' (raw deaths) or 'Rate' (per 100k)")
print("  val             — the death count or rate")
print("  evidence_strength — how strong the pollution-disease link is")
print(f"  {PM25_MEAN_COL:30s} — state annual mean PM2.5")
print("  join_quality    — filter to 'full' for reliable analysis")
print()

print("How to use this file in analysis (Phase 5):")
print("  # Get rows with full pollution+disease data, deaths as Number:")
print("  df = pd.read_csv('outputs/combined_master.csv')")
print("  df_clean = df[(df['join_quality'] == 'full') & (df['metric_id'] == 1)]")
print()
print("  # Correlate PM2.5 with COPD deaths across states:")
print("  copd = df_clean[df_clean['cause_name'] == 'Chronic obstructive pulmonary disease']")
print(f"  copd[['val', '{PM25_MEAN_COL}']].corr()")
print()

print("-" * 60)
print("NEXT: Phase 4 — AQI Calculation (phase_4_aqi.ipynb)")
print()
print("  Phase 4 will:")
print("    1. Load outputs/air_pollution_master.csv  (city x month)")
print("    2. Apply CPCB AQI breakpoint tables for PM2.5, PM10, NO2,")
print("       SO2, CO, NH3, Ozone")
print("    3. Compute sub-index per pollutant, final AQI = max sub-index")
print("    4. Assign CPCB category: Good/Satisfactory/Moderate/Poor/")
print("       Very Poor/Severe")
print("    5. Assign project 3-tier: Good(0-100)/Bad(101-300)/Worst(301+)")
print("    6. Save outputs/air_pollution_with_aqi.csv")
print("    7. Update combined_master.csv with state-year AQI summary columns")
print("=" * 60)

PHASE 3 COMPLETE — SUMMARY

File saved:
  outputs/combined_master.csv
      18,228 rows x 43 columns

What each row in combined_master.csv represents:
  One (state, year, disease cause, metric type) combination
  with pollution annual averages joined in from air_pollution_master.csv

Key columns for analysis:
  location_name   — state name (join key)
  year            — year (join key)
  cause_name      — disease (e.g. COPD, Ischemic heart disease)
  metric_name     — 'Number' (raw deaths) or 'Rate' (per 100k)
  val             — the death count or rate
  evidence_strength — how strong the pollution-disease link is
  PM2.5 (ug/m3)_mean             — state annual mean PM2.5
  join_quality    — filter to 'full' for reliable analysis

How to use this file in analysis (Phase 5):
  # Get rows with full pollution+disease data, deaths as Number:
  df = pd.read_csv('outputs/combined_master.csv')
  df_clean = df[(df['join_quality'] == 'full') & (df['metric_id'] == 1)]

  # Correlate PM2.5 with 